In [1]:
import pandas as pd
import numpy as np
import random
import sys
import os
from urllib.parse import urlparse, unquote
import re
import unicodedata
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, classification_report
import joblib

# Load dataset
df = pd.read_csv('../../ml/datasets/dataset_phishing.csv')
df['status'] = df['status'].map({'legitimate': 0, 'phishing': 1})

print("Dataset loaded:", df.shape)
print(df['status'].value_counts())

Dataset loaded: (11430, 89)
status
0    5715
1    5715
Name: count, dtype: int64


In [2]:
# Cyrillic lookalikes for Latin characters
cyrillic_map = {
    'a': 'а', 'e': 'е', 'o': 'о', 'p': 'р', 'c': 'с',
    'x': 'х', 'y': 'у', 'i': 'і', 's': 'ѕ', 'g': 'ɡ'
}

brands = [
    'paypal', 'google', 'facebook', 'apple', 'amazon',
    'microsoft', 'netflix', 'instagram', 'twitter', 'linkedin',
    'whatsapp', 'youtube', 'yahoo', 'gmail', 'outlook',
    'bankofamerica', 'chase', 'wellsfargo', 'dropbox', 'spotify'
]

tlds = ['.com', '.net', '.org', '.tk', '.ml', '.ga']
paths = ['/login', '/verify', '/secure', '/account', '/signin', '/update', '/confirm']

def make_cyrillic_url(brand):
    chars = list(brand)
    replaceable = [i for i, c in enumerate(chars) if c in cyrillic_map]
    if replaceable:
        for i in random.sample(replaceable, min(2, len(replaceable))):
            chars[i] = cyrillic_map[chars[i]]
    domain = ''.join(chars)
    tld = random.choice(tlds)
    path = random.choice(paths)
    return f'http://www.{domain}{tld}{path}'

def make_typo_url(brand):
    chars = list(brand)
    typo_map = {'o': '0', 'i': '1', 'e': '3', 'a': '@', 's': '5', 'l': '1'}
    replaceable = [i for i, c in enumerate(chars) if c in typo_map]
    if replaceable:
        for i in random.sample(replaceable, min(2, len(replaceable))):
            chars[i] = typo_map[chars[i]]
    domain = ''.join(chars)
    tld = random.choice(tlds)
    path = random.choice(paths)
    return f'http://www.{domain}{tld}{path}'

# Generate 3000 Cyrillic phishing URLs
cyrillic_urls = []
for _ in range(3000):
    brand = random.choice(brands)
    url = make_cyrillic_url(brand)
    cyrillic_urls.append({'url': url, 'status': 1})

# Generate 2000 typosquatting phishing URLs
typo_urls = []
for _ in range(2000):
    brand = random.choice(brands)
    url = make_typo_url(brand)
    typo_urls.append({'url': url, 'status': 1})

cyrillic_df = pd.concat([pd.DataFrame(cyrillic_urls), pd.DataFrame(typo_urls)], ignore_index=True)
print(f"Total phishing URLs generated: {len(cyrillic_df)}")
print(cyrillic_df['url'].head(10))

Total phishing URLs generated: 5000
0     http://www.bаnkofameriсa.ml/verify
1           http://www.аmаzon.tk/account
2         http://www.facebооk.ml/confirm
3        http://www.wеllsfаrgo.tk/secure
4            http://www.pауpal.ga/update
5           http://www.twіttеr.ga/update
6         http://www.lіnkedіn.org/secure
7         http://www.spоtіfy.net/confirm
8          http://www.facebооk.tk/secure
9    http://www.bankofamеriсa.com/update
Name: url, dtype: str


In [3]:
# Add backend to path so we can import features.py
sys.path.insert(0, '../../backend')
from features import extract_features

# Extract features from original dataset
print("Extracting features from original dataset...")
original_rows = []
for url, label in zip(df['url'], df['status']):
    feats = extract_features(url)
    feats['status'] = label
    original_rows.append(feats)

original_df = pd.DataFrame(original_rows)
print("Original dataset features extracted:", original_df.shape)

# Extract features from Cyrillic dataset
print("Extracting features from Cyrillic dataset...")
cyrillic_rows = []
for url, label in zip(cyrillic_df['url'], cyrillic_df['status']):
    feats = extract_features(url)
    feats['status'] = label
    cyrillic_rows.append(feats)

cyrillic_feat_df = pd.DataFrame(cyrillic_rows)
print("Cyrillic dataset features extracted:", cyrillic_feat_df.shape)

# Combine both datasets
combined_df = pd.concat([original_df, cyrillic_feat_df], ignore_index=True)
print("Combined dataset:", combined_df.shape)
print(combined_df['status'].value_counts())

Extracting features from original dataset...
Original dataset features extracted: (11430, 97)
Extracting features from Cyrillic dataset...
Cyrillic dataset features extracted: (5000, 97)
Combined dataset: (16430, 97)
status
1    10715
0     5715
Name: count, dtype: int64


In [4]:
# Generate 5000 legitimate URLs to balance the dataset
legit_domains = [
    'google.com', 'facebook.com', 'amazon.com', 'microsoft.com',
    'apple.com', 'netflix.com', 'linkedin.com', 'twitter.com',
    'instagram.com', 'youtube.com', 'yahoo.com', 'github.com',
    'stackoverflow.com', 'wikipedia.org', 'reddit.com', 'bbc.com',
    'cnn.com', 'nytimes.com', 'dropbox.com', 'spotify.com',
    'bbc.co.uk', 'theguardian.com', 'forbes.com', 'bloomberg.com',
    'techcrunch.com', 'medium.com', 'quora.com', 'pinterest.com',
    'ebay.com', 'etsy.com'
]

legit_paths = ['/', '/home', '/about', '/contact', '/search', '/help', '/news', '/products', '/services', '/blog']

legit_urls = []
for _ in range(5000):
    domain = random.choice(legit_domains)
    path = random.choice(legit_paths)
    url = f'https://www.{domain}{path}'
    legit_urls.append({'url': url, 'status': 0})

legit_df = pd.DataFrame(legit_urls)

# Extract features
legit_rows = []
for url, label in zip(legit_df['url'], legit_df['status']):
    feats = extract_features(url)
    feats['status'] = label
    legit_rows.append(feats)

legit_feat_df = pd.DataFrame(legit_rows)

# Add to combined dataset
combined_df = pd.concat([combined_df, legit_feat_df], ignore_index=True)
print("Balanced dataset:", combined_df.shape)
print(combined_df['status'].value_counts())

Balanced dataset: (21430, 97)
status
0    10715
1    10715
Name: count, dtype: int64


In [5]:
# Define URL-based features including new enhancements
url_features = [
    'length_url', 'length_hostname', 'ip', 'nb_dots', 'nb_hyphens',
    'nb_at', 'nb_qm', 'nb_and', 'nb_or', 'nb_eq', 'nb_underscore',
    'nb_tilde', 'nb_percent', 'nb_slash', 'nb_star', 'nb_colon',
    'nb_comma', 'nb_semicolumn', 'nb_dollar', 'nb_space', 'nb_www',
    'nb_com', 'nb_dslash', 'http_in_path', 'https_token',
    'ratio_digits_url', 'ratio_digits_host', 'punycode', 'port',
    'tld_in_path', 'tld_in_subdomain', 'abnormal_subdomain',
    'nb_subdomains', 'prefix_suffix', 'random_domain',
    'shortening_service', 'path_extension', 'nb_redirection',
    'nb_external_redirection', 'length_words_raw', 'char_repeat',
    'shortest_words_raw', 'shortest_word_host', 'shortest_word_path',
    'longest_words_raw', 'longest_word_host', 'longest_word_path',
    'avg_words_raw', 'avg_word_host', 'avg_word_path', 'phish_hints',
    'suspecious_tld', 'homograph', 'non_ascii', 'encoding_obfuscation',
    'typosquatting'
]

X = combined_df[url_features]
y = combined_df['status']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print("Training samples:", X_train.shape[0])
print("Testing samples:", X_test.shape[0])

# Train model
model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

y_pred = model.predict(X_test)
print("\nAccuracy:", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred))
print("Recall:", recall_score(y_test, y_pred))
print("\n", classification_report(y_test, y_pred))


Training samples: 17144
Testing samples: 4286

Accuracy: 0.9482034531031265
Precision: 0.9472455648926237
Recall: 0.9490177736202058

               precision    recall  f1-score   support

           0       0.95      0.95      0.95      2148
           1       0.95      0.95      0.95      2138

    accuracy                           0.95      4286
   macro avg       0.95      0.95      0.95      4286
weighted avg       0.95      0.95      0.95      4286



In [6]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV

# Try different parameters
param_grid = {
    'n_estimators': [100, 200, 300],
    'max_depth': [None, 20, 30],
    'min_samples_split': [2, 5],
    'min_samples_leaf': [1, 2]
}

rf = RandomForestClassifier(random_state=42)
grid_search = GridSearchCV(rf, param_grid, cv=3, scoring='accuracy', n_jobs=-1, verbose=1)
grid_search.fit(X_train, y_train)

print("Best parameters:", grid_search.best_params_)
print("Best accuracy:", grid_search.best_score_)

Fitting 3 folds for each of 36 candidates, totalling 108 fits
Best parameters: {'max_depth': 30, 'min_samples_leaf': 1, 'min_samples_split': 5, 'n_estimators': 200}
Best accuracy: 0.9493700654703193


In [7]:
# Train with best parameters
best_model = RandomForestClassifier(
    n_estimators=300,
    max_depth=30,
    min_samples_leaf=1,
    min_samples_split=2,
    random_state=42
)
best_model.fit(X_train, y_train)

y_best_pred = best_model.predict(X_test)
print("Accuracy:", accuracy_score(y_test, y_best_pred))
print("Precision:", precision_score(y_test, y_best_pred))
print("Recall:", recall_score(y_test, y_best_pred))
print("\n", classification_report(y_test, y_best_pred))

Accuracy: 0.9456369575361643
Precision: 0.9424059451927543
Recall: 0.9490177736202058

               precision    recall  f1-score   support

           0       0.95      0.94      0.95      2148
           1       0.94      0.95      0.95      2138

    accuracy                           0.95      4286
   macro avg       0.95      0.95      0.95      4286
weighted avg       0.95      0.95      0.95      4286



In [8]:
from xgboost import XGBClassifier

xgb_model = XGBClassifier(
    n_estimators=300,
    max_depth=6,
    learning_rate=0.1,
    random_state=42,
    eval_metric='logloss'
)
xgb_model.fit(X_train, y_train)

y_xgb_pred = xgb_model.predict(X_test)
print("Accuracy:", accuracy_score(y_test, y_xgb_pred))
print("Precision:", precision_score(y_test, y_xgb_pred))
print("Recall:", recall_score(y_test, y_xgb_pred))

Accuracy: 0.9461035930937938
Precision: 0.9499764039641341
Recall: 0.941534144059869


In [9]:
joblib.dump(best_model, '../../ml/trained/phishing_model.pkl')
joblib.dump(url_features, '../../ml/trained/feature_columns.pkl')
print("Model saved!")

Model saved!


In [10]:
# Try to push accuracy higher
better_model = RandomForestClassifier(
    n_estimators=500,
    max_depth=None,
    min_samples_split=2,
    min_samples_leaf=1,
    max_features='sqrt',
    random_state=42,
    n_jobs=-1
)
better_model.fit(X_train, y_train)

y_better_pred = better_model.predict(X_test)
print("Accuracy:", accuracy_score(y_test, y_better_pred))
print("Precision:", precision_score(y_test, y_better_pred))
print("Recall:", recall_score(y_test, y_better_pred))


Accuracy: 0.9463369108726085
Precision: 0.9420759962928638
Recall: 0.95088868101029


In [11]:
joblib.dump(better_model, '../../ml/trained/phishing_model.pkl')
joblib.dump(url_features, '../../ml/trained/feature_columns.pkl')
print("Final model saved!")

Final model saved!
